# 0. 套件匯入與環境設定


In [19]:
from pathlib import Path
import os
import warnings

notebook_dir = Path().resolve()
data_path = notebook_dir.parent / "data" / "secom.csv"
img_dir = notebook_dir / "img"
img_dir.mkdir(exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(notebook_dir / ".matplotlib"))
(notebook_dir / ".matplotlib").mkdir(exist_ok=True)
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")

import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Microsoft JhengHei"
plt.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings("ignore", category=UserWarning)

from imblearn.over_sampling import SMOTE
from scipy import stats as sps
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, balanced_accuracy_score, r2_score,
    mean_absolute_error, mean_squared_error
)
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.diagnostic import linear_reset, linear_rainbow, het_breuschpagan
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# 1. 資料讀取


In [20]:
secom_raw = pd.read_csv(data_path, sep="\t")
print(secom_raw.shape)
display(secom_raw.head())


(1567, 592)


,Time,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,x582,x583,x584,x585,x586,x587,x588,x589,x590,Pass/Fail
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


# 2. 初步前處理


In [21]:
def preprocess_secom(secom_df):
    X = secom_df.drop(columns=["Time", "Pass/Fail"])
    y = secom_df["Pass/Fail"].replace({-1: 0, 1: 1}).astype(int)

    na_counts = X.isna().sum()
    X = X.loc[:, na_counts <= 1000].copy()

    uniq_counts = X.nunique(dropna=False)
    X = X.loc[:, uniq_counts >= 10].copy()

    X = X.loc[:, ~X.T.duplicated(keep="first")].copy()

    imputer = KNNImputer(n_neighbors=10, weights="distance")
    X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

    scaler = StandardScaler()
    X_std = pd.DataFrame(scaler.fit_transform(X_imp), columns=X.columns, index=X.index)
    return X_std, y, imputer, scaler

X_std_initial, y_initial, imputer_initial, scaler_initial = preprocess_secom(secom_raw)
pca_initial = PCA(n_components=0.9)
X_pca_initial = pca_initial.fit_transform(X_std_initial)
X_pca_df = pd.DataFrame(
    X_pca_initial,
    columns=[f"pc{i}" for i in range(1, X_pca_initial.shape[1] + 1)],
    index=X_std_initial.index,
)
print(f"PCA 保留 {X_pca_initial.shape[1]} 個主成分")


PCA 保留 131 個主成分


# 3. PCA 離群樣本判斷


In [22]:
outlier = X_pca_df[(X_pca_df["pc1"] > 10) | (X_pca_df["pc2"] > 10)].index.tolist()
print(f"PCA 極端值有: {outlier}")
print(f"共 {len(outlier)} 個")


PCA 極端值有: [8, 10, 14, 15, 23, 24, 25, 33, 36, 44, 46, 47, 51, 54, 57, 58, 61, 72, 84, 91, 93, 96, 99, 143, 144, 163, 172, 183, 186, 195, 196, 275, 457, 466, 539, 634, 709, 800, 1397, 1427, 1458, 1489]
共 42 個


# 4. 排除離群樣本後重新前處理


In [23]:
secom = pd.read_csv(data_path, sep="\t")
secom = secom.drop(index=outlier).reset_index(drop=True)
X_std, y, imputer, scaler = preprocess_secom(secom)
print(X_std.shape)
print(y.value_counts().sort_index())


(1525, 459)
Pass/Fail
0    1430
1      95
Name: count, dtype: int64


# 5. SMOTE 後 PCA 與 KMeans 分群


In [24]:
smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy=1, k_neighbors=5)
X_smote, y_smote = smote.fit_resample(X_std, y)
X_smote_df = pd.DataFrame(X_smote, columns=X_std.columns)

pca = PCA(n_components=0.9, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_smote_df)
n_real = len(X_std)

assert np.allclose(
    X_smote_df.iloc[:n_real].drop(columns=["cluster"], errors="ignore").values,
    X_std.values,
    atol=1e-7,
    rtol=1e-7,
)
assert np.array_equal(np.asarray(y_smote)[:n_real], np.asarray(y))

kmeans = KMeans(n_clusters=2, n_init=10, max_iter=300, random_state=RANDOM_STATE)
cluster = kmeans.fit_predict(X_pca)
X_smote_df["cluster"] = cluster
print(pd.Series(cluster[:n_real]).value_counts().sort_index())


0    676
1    849
Name: count, dtype: int64


C:\Users\No\AppData\Local\Temp\ipykernel_4584\1205146053.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_smote_df["cluster"] = cluster


# 6. 建立真實樣本資料表


In [25]:
X_all_df = X_smote_df.drop(columns=["cluster"]).iloc[:n_real].copy()
c_all = X_smote_df.loc[:n_real - 1, "cluster"].to_numpy()
y_real = np.asarray(y)

assert len(X_all_df) == len(y_real) == len(c_all)
real_sample_df = pd.DataFrame({
    "original_index": X_all_df.index,
    "true_y": y_real,
    "cluster": c_all,
})
display(real_sample_df.groupby("cluster")["true_y"].agg(["count", "sum", "mean"]))


,count,sum,mean
cluster,,,
0,676,57,0.084320
1,849,38,0.044759


# 7. 各群模型訓練與 threshold 選擇


In [26]:
def make_xgb(params):
    return XGBClassifier(
        **params,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
    )

model_grids = [
    {"model_name": "M1", "params": {"n_estimators": 200, "max_depth": 2, "learning_rate": 0.03, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_weight": 1, "gamma": 0.0, "reg_lambda": 2.0, "reg_alpha": 0.0}},
    {"model_name": "M2", "params": {"n_estimators": 300, "max_depth": 2, "learning_rate": 0.02, "subsample": 0.80, "colsample_bytree": 0.70, "min_child_weight": 2, "gamma": 0.0, "reg_lambda": 3.0, "reg_alpha": 0.1}},
    {"model_name": "M3", "params": {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.03, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_weight": 1, "gamma": 0.0, "reg_lambda": 1.5, "reg_alpha": 0.0}},
    {"model_name": "M4", "params": {"n_estimators": 700, "max_depth": 3, "learning_rate": 0.015, "subsample": 0.70, "colsample_bytree": 0.70, "min_child_weight": 3, "gamma": 0.5, "reg_lambda": 4.0, "reg_alpha": 0.5}},
    {"model_name": "M5", "params": {"n_estimators": 500, "max_depth": 2, "learning_rate": 0.01, "subsample": 0.75, "colsample_bytree": 0.75, "min_child_weight": 3, "gamma": 0.2, "reg_lambda": 5.0, "reg_alpha": 0.5}},
    {"model_name": "M6", "params": {"n_estimators": 300, "max_depth": 4, "learning_rate": 0.02, "subsample": 0.70, "colsample_bytree": 0.70, "min_child_weight": 2, "gamma": 0.5, "reg_lambda": 4.0, "reg_alpha": 1.0}},
]

threshold_grid = np.arange(0.01, 1.00, 0.01)

RESEARCH_FINAL_SELECTION = {
    0: {"model_name": "M3", "threshold": 0.15},
    1: {"model_name": "M4", "threshold": 0.37},
}

def safe_auc(y_true, proba):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, proba))

def compute_binary_metrics(y_true, proba, threshold):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba, dtype=float)
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    false_alarm_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    recall = recall_score(y_true, pred, zero_division=0)
    precision = precision_score(y_true, pred, zero_division=0)
    gmean = float(np.sqrt(recall * specificity)) if recall >= 0 and specificity >= 0 else 0.0
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision,
        "recall": recall,
        "f1": f1_score(y_true, pred, zero_division=0),
        "specificity": specificity,
        "false_alarm_rate": false_alarm_rate,
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "gmean": gmean,
        "auc": safe_auc(y_true, proba),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def fit_with_smote(X_train, y_train, params, sampling_strategy=0.3):
    y_train = np.asarray(y_train).astype(int)
    counts = pd.Series(y_train).value_counts()
    if len(counts) < 2 or counts.min() < 2:
        raise ValueError("SMOTE 需要每個類別至少 2 筆樣本。")
    sm = SMOTE(
        sampling_strategy=sampling_strategy,
        k_neighbors=min(5, int(counts.min()) - 1),
        random_state=RANDOM_STATE,
    )
    X_sm, y_sm = sm.fit_resample(X_train, y_train)
    model = make_xgb(params)
    model.fit(X_sm, y_sm)
    return model

def select_best_threshold(y_true, oof_proba):
    rows = []
    for threshold in threshold_grid:
        row = compute_binary_metrics(y_true, oof_proba, threshold)
        row["threshold"] = float(threshold)
        rows.append(row)
    df = pd.DataFrame(rows)
    df = df.sort_values(
        ["f1", "recall", "precision", "threshold"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)
    return float(df.loc[0, "threshold"]), df.loc[0].to_dict(), df

cluster_splits = {}
cluster_model_artifacts = {}
selection_rows = []
fold_metric_rows = []
threshold_search_tables = []

for cluster_id in sorted(np.unique(c_all)):
    idx = np.where(c_all == cluster_id)[0]
    X_c = X_all_df.iloc[idx].copy()
    y_c = pd.Series(y_real[idx], index=X_c.index)

    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X_c, y_c, test_size=0.3, stratify=y_c, random_state=RANDOM_STATE
        )
    except ValueError as exc:
        raise ValueError(f"cluster {cluster_id} 無法進行 stratified train/test split，請檢查不良品數量。原始錯誤: {exc}")

    cluster_splits[int(cluster_id)] = {
        "X_train": X_train, "X_test": X_test,
        "y_train": y_train, "y_test": y_test,
    }

    min_class = int(pd.Series(y_train).value_counts().min())
    if min_class < 5:
        raise ValueError(f"cluster {cluster_id} training data 類別數不足，無法做 5-fold StratifiedKFold。")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    for spec in model_grids:
        model_name = spec["model_name"]
        params = spec["params"]
        oof_proba = np.zeros(len(X_train), dtype=float)
        fold_train_metrics = []
        fold_valid_metrics = []

        for fold_id, (tr_pos, va_pos) in enumerate(skf.split(X_train, y_train), start=1):
            X_tr = X_train.iloc[tr_pos]
            y_tr = y_train.iloc[tr_pos]
            X_va = X_train.iloc[va_pos]
            y_va = y_train.iloc[va_pos]

            model = fit_with_smote(X_tr, y_tr, params)
            train_proba = model.predict_proba(X_tr)[:, 1]
            valid_proba = model.predict_proba(X_va)[:, 1]
            oof_proba[va_pos] = valid_proba

            train_m = compute_binary_metrics(y_tr, train_proba, 0.5)
            valid_m = compute_binary_metrics(y_va, valid_proba, 0.5)
            train_m.update({"cluster": int(cluster_id), "model_name": model_name, "fold": fold_id, "split": "fold_train"})
            valid_m.update({"cluster": int(cluster_id), "model_name": model_name, "fold": fold_id, "split": "fold_valid"})
            fold_train_metrics.append(train_m)
            fold_valid_metrics.append(valid_m)
            fold_metric_rows.extend([train_m, valid_m])

        best_threshold, best_metrics, threshold_table = select_best_threshold(y_train, oof_proba)
        threshold_table["cluster"] = int(cluster_id)
        threshold_table["model_name"] = model_name
        threshold_search_tables.append(threshold_table)

        row = {"cluster": int(cluster_id), "model_name": model_name, "best_threshold": best_threshold}
        for key, val in best_metrics.items():
            if key != "threshold":
                row[f"oof_{key}"] = val

        train_df = pd.DataFrame(fold_train_metrics)
        valid_df = pd.DataFrame(fold_valid_metrics)
        for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
            row[f"mean_fold_train_{metric}"] = train_df[metric].mean()
            row[f"mean_fold_valid_{metric}"] = valid_df[metric].mean()

        selection_rows.append(row)
        cluster_model_artifacts[(int(cluster_id), model_name)] = {
            "oof_proba": oof_proba,
            "best_threshold": best_threshold,
            "best_oof_metrics": best_metrics,
            "threshold_table": threshold_table,
            "fold_train_metrics": fold_train_metrics,
            "fold_valid_metrics": fold_valid_metrics,
            "params": params,
        }

cluster_model_selection_df = pd.DataFrame(selection_rows)
cluster_model_selection_df = cluster_model_selection_df.sort_values(
    ["cluster", "oof_f1", "oof_recall", "oof_precision", "oof_gmean"],
    ascending=[True, False, False, False, False],
).reset_index(drop=True)
fold_metrics_df = pd.DataFrame(fold_metric_rows)
threshold_search_df = pd.concat(threshold_search_tables, ignore_index=True)
display(cluster_model_selection_df)


,cluster,model_name,best_threshold,oof_accuracy,oof_precision,oof_recall,oof_f1,oof_specificity,oof_false_alarm_rate,oof_balanced_accuracy,...,mean_fold_train_accuracy,mean_fold_valid_accuracy,mean_fold_train_precision,mean_fold_valid_precision,mean_fold_train_recall,mean_fold_valid_recall,mean_fold_train_f1,mean_fold_valid_f1,mean_fold_train_auc,mean_fold_valid_auc
0,0,M3,0.14,0.856237,0.274194,0.425000,0.333333,0.896074,0.103926,0.660537,...,1.000000,0.913326,1.000000,0.350000,1.000000,0.100,1.000000,0.153333,1.000000,0.744610
1,0,M4,0.20,0.860465,0.275862,0.400000,0.326531,0.903002,0.096998,0.651501,...,0.998943,0.911198,1.000000,0.416667,0.987500,0.100,0.993651,0.154141,1.000000,0.753909
2,0,M1,0.18,0.822410,0.238095,0.500000,0.322581,0.852194,0.147806,0.676097,...,0.991014,0.909093,1.000000,0.373333,0.893750,0.125,0.943594,0.183497,0.999007,0.753154
3,0,M6,0.21,0.864693,0.277778,0.375000,0.319149,0.909931,0.090069,0.642465,...,0.995773,0.906965,1.000000,0.116667,0.950000,0.050,0.974086,0.069697,0.999946,0.775194
4,0,M5,0.28,0.862579,0.272727,0.375000,0.315789,0.907621,0.092379,0.641311,...,0.979385,0.909071,0.991667,0.406667,0.762500,0.100,0.861892,0.151577,0.994314,0.768645
5,0,M2,0.16,0.778013,0.207207,0.575000,0.304636,0.796767,0.203233,0.685883,...,0.989426,0.911221,1.000000,0.400000,0.875000,0.100,0.932740,0.153535,0.998249,0.752509
6,1,M3,0.19,0.929293,0.142857,0.111111,0.125000,0.968254,0.031746,0.539683,...,1.000000,0.951189,1.000000,0.000000,1.000000,0.000,1.000000,0.000000,1.000000,0.552119
7,1,M1,0.23,0.897306,0.095238,0.148148,0.115942,0.932981,0.067019,0.540564,...,0.995789,0.947828,1.000000,0.000000,0.907792,0.000,0.950488,0.000000,1.000000,0.519584
8,1,M5,0.26,0.893939,0.090909,0.148148,0.112676,0.929453,0.070547,0.538801,...,0.987794,0.949509,1.000000,0.000000,0.730736,0.000,0.843806,0.000000,0.998574,0.537354
9,1,M2,0.22,0.892256,0.088889,0.148148,0.111111,0.927690,0.072310,0.537919,...,0.994527,0.944467,1.000000,0.000000,0.879654,0.000,0.935330,0.000000,0.999959,0.524220


# 8. Testing data 保留樣本評估


In [27]:
final_models = {}
final_thresholds = {}
final_model_names = {}
final_model_params = {}
test_rows = []

for cluster_id in sorted(np.unique(c_all)):
    cluster_table = cluster_model_selection_df[cluster_model_selection_df["cluster"] == int(cluster_id)].copy()
    cv_best = cluster_table.sort_values(
        ["oof_f1", "oof_recall", "oof_precision", "oof_gmean"],
        ascending=[False, False, False, False],
    ).iloc[0]

    # Keep the full CV comparison; use the confirmed research final setting for testing and local behavior.
    research_selection = RESEARCH_FINAL_SELECTION[int(cluster_id)]
    model_name = research_selection["model_name"]
    artifact = cluster_model_artifacts[(int(cluster_id), model_name)]
    threshold = float(research_selection["threshold"])
    params = artifact["params"]
    if cv_best["model_name"] != model_name or not np.isclose(float(cv_best["best_threshold"]), threshold):
        print(
            f"cluster {cluster_id}: CV best is {cv_best['model_name']} "
            f"(threshold={float(cv_best['best_threshold']):.2f}); "
            f"using research final setting {model_name} (threshold={threshold:.2f})."
        )

    split = cluster_splits[int(cluster_id)]
    X_train, y_train = split["X_train"], split["y_train"]
    X_test, y_test = split["X_test"], split["y_test"]

    final_model = fit_with_smote(X_train, y_train, params)
    final_models[int(cluster_id)] = final_model
    final_thresholds[int(cluster_id)] = threshold
    final_model_names[int(cluster_id)] = model_name
    final_model_params[int(cluster_id)] = params

    test_proba = final_model.predict_proba(X_test)[:, 1]
    test_m = compute_binary_metrics(y_test, test_proba, threshold)
    test_row = {
        "cluster": int(cluster_id),
        "selected_model": model_name,
        "selected_threshold": threshold,
        "n_train": int(len(y_train)),
        "n_test": int(len(y_test)),
        "train_positive": int(np.asarray(y_train).sum()),
        "test_positive": int(np.asarray(y_test).sum()),
    }
    for key, val in test_m.items():
        test_row[f"test_{key}" if key not in ["tn", "fp", "fn", "tp"] else key] = val
    test_rows.append(test_row)

cluster_test_result_df = pd.DataFrame(test_rows)
display(cluster_test_result_df)


cluster 0: CV best is M3 (threshold=0.14); using research final setting M3 (threshold=0.15).
cluster 1: CV best is M3 (threshold=0.19); using research final setting M4 (threshold=0.37).


,cluster,selected_model,selected_threshold,n_train,n_test,train_positive,test_positive,test_accuracy,test_precision,test_recall,test_f1,test_specificity,test_false_alarm_rate,test_balanced_accuracy,test_gmean,test_auc,tn,fp,fn,tp
0,0,M3,0.15,473,203,40,17,0.832512,0.130435,0.176471,0.15,0.892473,0.107527,0.534472,0.396857,0.619861,166,20,14,3
1,1,M4,0.37,594,255,27,11,0.956863,0.000000,0.000000,0.00,1.000000,0.000000,0.500000,0.000000,0.717213,244,0,11,0


# 9. Overfitting 檢查


In [28]:
overfit_rows = []
for _, test_row in cluster_test_result_df.iterrows():
    cluster_id = int(test_row["cluster"])
    model_name = test_row["selected_model"]
    sel = cluster_model_selection_df[
        (cluster_model_selection_df["cluster"] == cluster_id) &
        (cluster_model_selection_df["model_name"] == model_name)
    ].iloc[0]
    row = {
        "cluster": cluster_id,
        "selected_model": model_name,
        "selected_threshold": float(test_row["selected_threshold"]),
    }
    for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
        row[f"mean_fold_train_{metric}"] = float(sel[f"mean_fold_train_{metric}"])
        row[f"oof_{metric}"] = float(sel[f"oof_{metric}"])
        row[f"test_{metric}"] = float(test_row[f"test_{metric}"])
    row["train_oof_f1_gap"] = row["mean_fold_train_f1"] - row["oof_f1"]
    row["oof_test_f1_gap"] = row["oof_f1"] - row["test_f1"]
    row["train_oof_auc_gap"] = row["mean_fold_train_auc"] - row["oof_auc"]
    row["oof_test_auc_gap"] = row["oof_auc"] - row["test_auc"]
    overfit_rows.append(row)

overfit_check_df = pd.DataFrame(overfit_rows)
display(overfit_check_df)


,cluster,selected_model,selected_threshold,mean_fold_train_accuracy,oof_accuracy,test_accuracy,mean_fold_train_precision,oof_precision,test_precision,mean_fold_train_recall,...,mean_fold_train_f1,oof_f1,test_f1,mean_fold_train_auc,oof_auc,test_auc,train_oof_f1_gap,oof_test_f1_gap,train_oof_auc_gap,oof_test_auc_gap
0,0,M3,0.15,1.0,0.856237,0.832512,1.0,0.274194,0.130435,1.0,...,1.0,0.333333,0.15,1.0,0.744861,0.619861,0.666667,0.183333,0.255139,0.125001
1,1,M4,0.37,1.0,0.646465,0.956863,1.0,0.057971,0.000000,1.0,...,1.0,0.102564,0.00,1.0,0.550069,0.717213,0.897436,0.102564,0.449931,-0.167145


# 10. Behavior final model 與局部刻劃用 hatp


In [29]:
for cluster_id, expected in RESEARCH_FINAL_SELECTION.items():
    expected_params = next(m["params"] for m in model_grids if m["model_name"] == expected["model_name"])
    final_model_names[cluster_id] = expected["model_name"]
    final_thresholds[cluster_id] = expected["threshold"]
    final_model_params[cluster_id] = expected_params

behavior_models = {}
behavior_proba = {}
behavior_rows = []
region_rows = []
high_risk_cutoff = 0.95
boundary_width = 0.05

for cluster_id in sorted(np.unique(c_all)):
    cluster_mask = np.asarray(c_all) == int(cluster_id)
    cluster_positions = np.where(cluster_mask)[0]
    X_cluster_all = X_all_df.iloc[cluster_positions].copy()
    y_cluster_all = np.asarray(y_real)[cluster_positions].astype(int)

    selected_model = final_model_names[int(cluster_id)]
    selected_threshold = float(final_thresholds[int(cluster_id)])
    params = final_model_params[int(cluster_id)]

    behavior_model = fit_with_smote(X_cluster_all, y_cluster_all, params)
    behavior_models[int(cluster_id)] = behavior_model

    # 此處 hatp 用於局部刻劃，描述 final model 的內部預測行為，
    # 不作為 testing performance 或 threshold 選擇依據。
    hatp = behavior_model.predict_proba(X_cluster_all)[:, 1]
    behavior_proba[int(cluster_id)] = pd.Series(hatp, index=X_cluster_all.index)

    # 決策邊界區以 training OOF probability 選出的 threshold 為中心。
    boundary_lower = selected_threshold - boundary_width
    boundary_upper = selected_threshold + boundary_width
    decision_boundary_region = (hatp >= boundary_lower) & (hatp <= boundary_upper)
    high_risk_region = hatp >= high_risk_cutoff

    print(f"cluster {cluster_id} high risk samples: {int(high_risk_region.sum())}")

    for original_index, true_y, proba_value, is_high, is_boundary in zip(
        X_cluster_all.index, y_cluster_all, hatp, high_risk_region, decision_boundary_region
    ):
        behavior_rows.append({
            "original_index": original_index,
            "true_y": int(true_y),
            "hatp": float(proba_value),
            "cluster": int(cluster_id),
            "selected_model": selected_model,
            "selected_threshold": selected_threshold,
            "high_risk_region": bool(is_high),
            "decision_boundary_region": bool(is_boundary),
        })

    boundary_y = y_cluster_all[decision_boundary_region]
    high_y = y_cluster_all[high_risk_region]
    region_rows.append({
        "cluster": int(cluster_id),
        "selected_model": selected_model,
        "selected_threshold": selected_threshold,
        "boundary_lower": boundary_lower,
        "boundary_upper": boundary_upper,
        "n_boundary": int(decision_boundary_region.sum()),
        "boundary_positive": int(boundary_y.sum()) if len(boundary_y) else 0,
        "boundary_positive_rate": float(boundary_y.mean()) if len(boundary_y) else np.nan,
        "high_risk_cutoff": high_risk_cutoff,
        "n_high_risk": int(high_risk_region.sum()),
        "high_risk_positive": int(high_y.sum()) if len(high_y) else 0,
        "high_risk_positive_rate": float(high_y.mean()) if len(high_y) else np.nan,
    })

df_behavior_proba = pd.DataFrame(behavior_rows).sort_values(["cluster", "original_index"]).reset_index(drop=True)
region_summary_df = pd.DataFrame(region_rows)

p_hat_behavior = np.full(len(X_all_df), np.nan, dtype=float)
for cluster_id, proba_s in behavior_proba.items():
    p_hat_behavior[X_all_df.index.get_indexer(proba_s.index)] = proba_s.to_numpy()
p_hat_oof = p_hat_behavior.copy()

blackbox_by_c = behavior_models
blackbox_by_0 = {0: behavior_models[0]}
blackbox_by_1 = {1: behavior_models[1]}

display(region_summary_df)
display(df_behavior_proba.head())


cluster 0 high risk samples: 3
cluster 1 high risk samples: 3


,cluster,selected_model,selected_threshold,boundary_lower,boundary_upper,n_boundary,boundary_positive,boundary_positive_rate,high_risk_cutoff,n_high_risk,high_risk_positive,high_risk_positive_rate
0,0,M3,0.15,0.10,0.20,22,0,0.0,0.95,3,3,1.0
1,1,M4,0.37,0.32,0.42,1,1,1.0,0.95,3,3,1.0


,original_index,true_y,hatp,cluster,selected_model,selected_threshold,high_risk_region,decision_boundary_region
0,0,0,0.019355,0,M3,0.15,False,False
1,1,0,0.022467,0,M3,0.15,False,False
2,2,1,0.865441,0,M3,0.15,False,False
3,3,0,0.054326,0,M3,0.15,False,False
4,4,0,0.022647,0,M3,0.15,False,False


In [30]:
df_behavior_proba[df_behavior_proba['original_index']==392]

,original_index,true_y,hatp,cluster,selected_model,selected_threshold,high_risk_region,decision_boundary_region
387,392,1,0.962766,0,M3,0.15,True,False


# 11. 局部區域選定


In [31]:
print("高度不良品預測區：hatp >= 0.95")
display(df_behavior_proba.groupby(["cluster", "high_risk_region"]).size().rename("n").reset_index())
for cluster_id in sorted(np.unique(c_all)):
    view = df_behavior_proba[(df_behavior_proba["cluster"] == cluster_id) & (df_behavior_proba["high_risk_region"])].sort_values("hatp", ascending=False)
    print(f"cluster {cluster_id}: n={len(view)}")
    display(view[["original_index", "true_y", "hatp", "cluster", "selected_threshold", "high_risk_region"]])

print("決策邊界區：selected threshold ± 0.05")
display(df_behavior_proba.groupby(["cluster", "decision_boundary_region"]).size().rename("n").reset_index())
for cluster_id in sorted(np.unique(c_all)):
    view = df_behavior_proba[(df_behavior_proba["cluster"] == cluster_id) & (df_behavior_proba["decision_boundary_region"])].sort_values("hatp", ascending=False)
    print(f"cluster {cluster_id}: n={len(view)}")
    display(view[["original_index", "true_y", "hatp", "cluster", "selected_threshold", "decision_boundary_region"]])


高度不良品預測區：hatp >= 0.95


,cluster,high_risk_region,n
0,0,False,673
1,0,True,3
2,1,False,846
3,1,True,3


cluster 0: n=3


,original_index,true_y,hatp,cluster,selected_threshold,high_risk_region
387,392,1,0.962766,0,0.15,True
209,210,1,0.958080,0,0.15,True
288,289,1,0.950744,0,0.15,True


cluster 1: n=3


...

,cluster,decision_boundary_region,n
0,0,False,654
1,0,True,22
2,1,False,848
3,1,True,1


cluster 0: n=22


,original_index,true_y,hatp,cluster,selected_threshold,decision_boundary_region
175,176,0,0.154267,0,0.15,True
108,109,0,0.150114,0,0.15,True
19,20,0,0.140902,0,0.15,True
133,134,0,0.132783,0,0.15,True
247,248,0,0.132226,0,0.15,True
197,198,0,0.130636,0,0.15,True
139,140,0,0.130469,0,0.15,True
104,105,0,0.118838,0,0.15,True
283,284,0,0.117755,0,0.15,True
183,184,0,0.117276,0,0.15,True


cluster 1: n=1


,original_index,true_y,hatp,cluster,selected_threshold,decision_boundary_region
1151,1147,1,0.360926,1,0.37,True


# 12. 中心點候選者設定


In [32]:
def get_center_candidates(cluster_id, region):
    region_col_map = {
        "high_risk": "high_risk_region",
        "decision_boundary": "decision_boundary_region",
        "boundary": "decision_boundary_region",
    }
    if region not in region_col_map:
        raise ValueError(f"Unknown local region: {region}")
    region_col = region_col_map[region]
    return df_behavior_proba[
        (df_behavior_proba["cluster"] == int(cluster_id)) &
        (df_behavior_proba[region_col].astype(bool))
    ].copy()

def get_center_candidate_positions(cluster_id, region):
    candidates = get_center_candidates(cluster_id, region)
    idx_c = np.where(np.asarray(c_all) == int(cluster_id))[0]
    cluster_index = list(X_all_df.iloc[idx_c].index)
    index_to_local_pos = {idx_value: pos for pos, idx_value in enumerate(cluster_index)}
    positions = [
        index_to_local_pos[idx_value]
        for idx_value in candidates["original_index"].tolist()
        if idx_value in index_to_local_pos
    ]
    if len(positions) != len(candidates):
        print(f"Warning: cluster {cluster_id} {region} has {len(candidates) - len(positions)} unmapped candidates.")
    if int(cluster_id) == 1 and region in {"decision_boundary", "boundary"} and len(positions) <= 1:
        print(f"Warning: cluster {cluster_id} decision_boundary has only {len(positions)} center candidate; local model may be unstable.")
    return np.asarray(positions, dtype=int)

for cluster_id in [0, 1]:
    for region_name in ["high_risk", "decision_boundary"]:
        center_candidates = get_center_candidates(cluster_id, region_name)
        print(cluster_id, region_name, len(center_candidates))


0 high_risk 3
0 decision_boundary 22
1 high_risk 3
1 decision_boundary 1


# 13. Local surrogate model 建立


In [33]:
def _sample_offsets_block(q, k, n_block, center_first=True, rng=None):
    """
    產生 (n_block, q) 的整數位移 offsets ∈ [-k, k]
    center_first=True：第一筆固定為全 0
    """
    if rng is None:
        rng = np.random.default_rng(0)

    offsets = rng.integers(-k, k + 1, size=(n_block, q), dtype=np.int16).astype(np.float32)
    if center_first and n_block > 0:
        offsets[0, :] = 0.0
    return offsets


def _ols_from_sufficient_stats(Sxx, Sxy, Syy, sum_y, n, q):
    """
    由 sufficient statistics（X'X, X'y, y'y, sum(y)）計算 OLS 係數與統計量
    用累積量求 OLS 係數與統計量
    輸出包含 beta / se / t / p / r2 / aic / bic / df_resi
    """
    p_params = q + 1
    df_resid = n - p_params
    out = {}

    try:
        beta = np.linalg.solve(Sxx, Sxy)
    except np.linalg.LinAlgError:
        beta = np.full(p_params, np.nan, dtype=np.float64)

    if np.all(np.isfinite(beta)):
        SSE = float(Syy - 2.0 * beta @ Sxy + beta @ (Sxx @ beta))
    else:
        SSE = np.nan

    ybar = float(sum_y / n) if n > 0 else np.nan
    TSS = float(Syy - n * (ybar**2)) if n > 0 else np.nan
    sigma2 = float(SSE / df_resid) if (df_resid > 0 and np.isfinite(SSE)) else np.nan

    try:
        Sxx_inv = np.linalg.inv(Sxx)
    except np.linalg.LinAlgError:
        Sxx_inv = np.full_like(Sxx, np.nan, dtype=np.float64)

    if np.isfinite(sigma2) and np.all(np.isfinite(Sxx_inv)):
        cov = sigma2 * Sxx_inv
        se = np.sqrt(np.diag(cov))
        tval = beta / se
        pval = 2.0 * sps.t.sf(np.abs(tval), df=df_resid) if df_resid>0 else np.full_like(tval, np.nan)
    else:
        se = np.full(p_params, np.nan, dtype=np.float64)
        tval = np.full(p_params, np.nan, dtype=np.float64)
        pval = np.full(p_params, np.nan, dtype=np.float64)

    r2 = (1.0 - SSE / TSS) if (np.isfinite(SSE) and np.isfinite(TSS) and TSS > 0) else np.nan

    kparam = p_params
    if n > 0 and np.isfinite(SSE) and SSE > 0:
        ll_term = n * np.log(SSE / n)
        aic = float(ll_term + 2.0 * kparam)
        bic = float(ll_term + np.log(n) * kparam)
    else:
        aic = np.nan
        bic = np.nan

    out["beta"] = beta
    out["SSE"] = SSE
    out["r2"] = float(r2) if np.isfinite(r2) else np.nan
    out["se"] = se
    out["t"] = tval
    out["p"] = pval
    out["aic"] = aic
    out["bic"] = bic
    out["df_resid"] = int(df_resid)
    return out


def local_surrogate_block_sample_ols(
    x0,
    blackbox_model,
    block_cols,
    delta,
    k,
    scaler=None,
    n_block=800_000,
    rng_seed=42,
    store_yhat=False,
    store_yblackbox=False,
):
    """
    在 x0 附近抽樣 n_block 個格點，丟進黑箱得到 y_hat
    一次只用 block_cols 的 q 維特徵做 OLS，輸出係數與 p-value
    """
    x0 = np.asarray(x0, np.float32).reshape(1, -1)

    if scaler is not None:
        x0_scaled = scaler.transform(x0).astype(np.float32, copy=False)
    else:
        x0_scaled = x0.copy()

    p_model = x0_scaled.shape[1]
    block_cols = np.asarray(block_cols, dtype=np.int64)
    q = int(block_cols.size)
    if q < 1:
        raise ValueError("block_cols 至少 1 維")
    if k < 0:
        raise ValueError("k must be >= 0")
    if store_yhat or store_yblackbox:
        raise ValueError("store_yhat/store_yblackbox 在大 n 下不可用（請關掉）。")

    rng = np.random.default_rng(rng_seed)

    offsets_q = _sample_offsets_block(q, k, n_block, center_first=True, rng=rng)
    X_block = (np.float32(delta) * offsets_q).astype(np.float32, copy=False)

    Zs = np.empty((n_block, p_model), dtype=np.float32)
    Zs[:] = x0_scaled
    Zs[:, block_cols] += X_block

    if scaler is not None:
        Z = scaler.inverse_transform(Zs).astype(np.float32, copy=False)
    else:
        Z = Zs

    if hasattr(blackbox_model, "predict_proba"):
        y = blackbox_model.predict_proba(Z)[:, 1]
    else:
        y = blackbox_model.predict(Z)

    y = np.asarray(y, dtype=np.float32).reshape(-1)
    if y.size != n_block:
        raise ValueError("blackbox 輸出長度與 n_block 不一致")

    Sxx = np.zeros((q + 1, q + 1), dtype=np.float64)
    Sxy = np.zeros((q + 1,), dtype=np.float64)

    Sxx[0, 0] = float(n_block)
    sx = X_block.sum(axis=0, dtype=np.float64)
    Sxx[0, 1:] = sx
    Sxx[1:, 0] = sx
    Sxx[1:, 1:] = (X_block.T @ X_block).astype(np.float64, copy=False)

    sy = float(y.sum(dtype=np.float64))
    Sxy[0] = sy
    Sxy[1:] = (X_block.T @ y).astype(np.float64, copy=False)

    Syy = float(y @ y)
    st = _ols_from_sufficient_stats(Sxx, Sxy, Syy, sy, int(n_block), q)

    info = {
        "n": int(n_block),
        "q": int(q),
        "k": int(k),
        "delta": float(delta),
        "block_cols": block_cols.tolist(),
        "coef_block": st["beta"][1:].copy(),
        "intercept": float(st["beta"][0]),
        "se_block": st["se"][1:].copy(),
        "t_block": st["t"][1:].copy(),
        "p_block": st["p"][1:].copy(),
        "intercept_p": float(st["p"][0]) if np.isfinite(st["p"][0]) else np.nan,
        "r2": float(st["r2"]),
        "aic": float(st["aic"]),
        "bic": float(st["bic"]),
        "df_resid": int(st["df_resid"]),
        "rng_seed": int(rng_seed),
    }
    return info


def pick_x0_indices(p, target, topk, min_p=None, candidate_idx=None):
    """
    依 |p - target| 選出最接近 target 的 topk 索引，可限定候選中心池。
    """
    p = np.asarray(p).reshape(-1)
    if candidate_idx is None:
        idx = np.arange(len(p))
    else:
        idx = np.asarray(candidate_idx, dtype=int).reshape(-1)
        idx = idx[(idx >= 0) & (idx < len(p))]
    if min_p is not None:
        idx = idx[p[idx] >= min_p]
    if idx.size == 0:
        return idx
    order = np.argsort(np.abs(p[idx] - target))
    return idx[order[:min(topk, idx.size)]]


def run_Nround_fullp_screening(
    x0,
    blackbox_model,
    feature_names,
    thresholds,
    k=3,
    delta=0.1,
    scaler=None,
    n_block=800_000,
    base_seed=42,
):
    """
    對單一 (x0, blackbox) 做多輪抽樣 OLS：
    每輪抽 n_block 個格點、用 active_cols 做 OLS，依門檻縮小 active_cols。
    回傳每輪的統計表 df_all 與最終保留欄位 final_cols。
    """
    p = len(feature_names)
    active_cols = np.arange(p, dtype=np.int64)
    all_round_dfs = []

    for r, thr in enumerate(thresholds, start=1):
        if active_cols.size == 0:
            break

        seed = int(base_seed + r * 100000)

        info = local_surrogate_block_sample_ols(
            x0=x0,
            blackbox_model=blackbox_model,
            block_cols=active_cols,
            delta=delta,
            k=k,
            scaler=scaler,
            n_block=n_block,
            rng_seed=seed,
            store_yhat=False,
            store_yblackbox=False,
        )

        rows = []
        for t, col in enumerate(active_cols):
            rows.append({
                "feature": feature_names[int(col)],
                "feature_idx": int(col),
                "coef": float(info["coef_block"][t]),
                "abs_coef": float(abs(info["coef_block"][t])),
                "t": float(info["t_block"][t]),
                "p": float(info["p_block"][t]),
                "r2_model": float(info["r2"]),
                "df_resid": int(info["df_resid"]),
                "rng_seed": int(info["rng_seed"]),
                "n_block": int(info["n"]),
                "round_id": int(r),
                "threshold": float(thr),
                "q_used": int(info["q"]),
            })

        df_r = pd.DataFrame(rows)
        if len(df_r):
            df_r = df_r.sort_values(["p", "abs_coef"], ascending=[True, False]).reset_index(drop=True)
            df_r["rank_p"] = np.arange(1, len(df_r) + 1)

        all_round_dfs.append(df_r)

        pass_cols = df_r.loc[df_r["p"] <= thr, "feature_idx"].unique()
        active_cols = np.asarray(pass_cols, dtype=np.int64)

    df_all = pd.concat(all_round_dfs, ignore_index=True) if len(all_round_dfs) else pd.DataFrame()
    return df_all, active_cols

# ===== (A) 最小新增：用 final features 在 x0 周圍抽大量格點、分批累積 sufficient stats =====

def local_surrogate_fixed_features_bigN(
    x0,
    blackbox_model,
    block_cols,          # final features indices
    delta,
    k,
    scaler=None,
    n_total=2_000_000,
    rng_seed=42,
    center_first=True,
):
    """
    固定特徵集合 block_cols (=final features)，在 x0 周圍一次抽 n_total 格點，
    以黑箱輸出 y_hat 做目標，對 X_block (=delta*offsets) 做 OLS：
        y_hat ~ [1, X_block]
    """
    x0 = np.asarray(x0, np.float32).reshape(1, -1)

    if scaler is not None:
        x0_scaled = scaler.transform(x0).astype(np.float32, copy=False)
    else:
        x0_scaled = x0.copy()

    p_model = x0_scaled.shape[1]
    block_cols = np.asarray(block_cols, dtype=np.int64)
    q = int(block_cols.size)
    if q < 1:
        raise ValueError("block_cols 至少 1 維")
    if k < 0:
        raise ValueError("k must be >= 0")
    if n_total < 1:
        raise ValueError("n_total must be >= 1")

    rng = np.random.default_rng(int(rng_seed))

    # 1) 一次生成 offsets / X_block（只在 q 維）
    offsets_q = _sample_offsets_block(
        q, k, int(n_total),
        center_first=bool(center_first),
        rng=rng
    )  # (n_total, q) float32
    X_block = (np.float32(delta) * offsets_q).astype(np.float32, copy=False)

    # 2) 一次組出完整 Z（n_total, p_model）再丟黑箱
    Zs = np.empty((int(n_total), p_model), dtype=np.float32)
    Zs[:] = x0_scaled
    Zs[:, block_cols] += X_block

    if scaler is not None:
        Z = scaler.inverse_transform(Zs).astype(np.float32, copy=False)
    else:
        Z = Zs

    if hasattr(blackbox_model, "predict_proba"):
        y = blackbox_model.predict_proba(Z)[:, 1]
    else:
        y = blackbox_model.predict(Z)

    y = np.asarray(y, dtype=np.float32).reshape(-1)
    if y.size != int(n_total):
        raise ValueError("blackbox 輸出長度與 n_total 不一致")
    
    Z_sub = X_block.astype(np.float64, copy=False)

    # 3) sufficient stats（一次算完）
    Sxx = np.zeros((q + 1, q + 1), dtype=np.float64)
    Sxy = np.zeros((q + 1,), dtype=np.float64)

    Sxx[0, 0] = float(n_total)
    sx = Z_sub.sum(axis=0, dtype=np.float64)
    Sxx[0, 1:] = sx
    Sxx[1:, 0] = sx
    Sxx[1:, 1:] = (Z_sub.T @ Z_sub).astype(np.float64, copy=False)

    sy = float(y.sum(dtype=np.float64))
    Sxy[0] = sy
    Sxy[1:] = (Z_sub.T @ y).astype(np.float64, copy=False)

    Syy = float(y @ y)
    st = _ols_from_sufficient_stats(Sxx, Sxy, Syy, sy, int(n_total), q)

    # 4) 用同一批建模點，逐點算 local surrogate 的預測值
    y_local = predict_local_surrogate_from_fullX(
        X_full=Zs,
        x0_center=x0_scaled.reshape(-1),
        block_cols=block_cols,
        intercept=float(st["beta"][0]),
        coef_block=st["beta"][1:],
    )

    # 5) 用逐點版重新計算 R^2
    r2_direct = float(r2_score(y.astype(np.float64), y_local))

    info = {
        "n": int(n_total),
        "q": int(q),
        "k": int(k),
        "delta": float(delta),
        "block_cols": block_cols.tolist(),
        "intercept": float(st["beta"][0]),
        "coef_block": st["beta"][1:].copy(),
        "se_block": st["se"][1:].copy(),
        "t_block": st["t"][1:].copy(),
        "p_block": st["p"][1:].copy(),
        "intercept_p": float(st["p"][0]) if np.isfinite(st["p"][0]) else np.nan,
        "r2": r2_direct,
        "aic": float(st["aic"]),
        "bic": float(st["bic"]),
        "df_resid": int(st["df_resid"]),
        "rng_seed": int(rng_seed),
        "x0_center": x0_scaled.reshape(-1).copy(),   # 此局部模型所對應的中心點（標準化空間）
        "X_local_full": Zs.copy(),                   # 完整局部資料集（標準化空間）
        "X_block_shift": X_block.copy(),             # block_cols 上的相對位移
        "y_blackbox": y.copy(),
    }
    return info


def predict_local_surrogate_from_fullX(X_full, x0_center, block_cols, intercept, coef_block):
    """
    用某群的局部線性模型 g_c 預測任意完整特徵矩陣 X_full 上的值
    X_full 必須與 x0_center 在同一空間（你這裡都是標準化空間）
    """
    X_full = np.asarray(X_full, dtype=np.float64)
    x0_center = np.asarray(x0_center, dtype=np.float64).reshape(-1)
    block_cols = np.asarray(block_cols, dtype=int)

    X_shift = X_full[:, block_cols] - x0_center[block_cols]
    y_pred = float(intercept) + X_shift @ np.asarray(coef_block, dtype=np.float64)
    return np.asarray(y_pred, dtype=np.float64)


def eval_surrogate(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    return {
        "r2": float(r2_score(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "corr": float(np.corrcoef(y_true, y_pred)[0, 1]),
    }


In [36]:
LOCAL_B = 50
LOCAL_N_BLOCK = 1_000_000
LOCAL_N_TOTAL = 2_000_000
LOCAL_K = 2
LOCAL_DELTA = 0.1
LOCAL_THRESHOLDS = (0.01, 0.005, 0.001, 0.0005, 0.0001)

local_model_summary_rows = []
local_model_coef_tables = {}
local_model_coef_rows = []
local_screening_tables = {}


def _display_value(values):
    values = list(values)
    if len(values) == 0:
        return np.nan
    if len(values) == 1:
        return values[0]
    return str(values)


def run_screening_for_local_model(cluster_id, region, topk, target, target_min_p=None, base_seed=42):
    feature_names = list(X_all_df.columns)
    idx_c = np.where(c_all == int(cluster_id))[0]
    p_hat_c = np.asarray(p_hat_oof)[idx_c]
    candidate_idx = get_center_candidate_positions(cluster_id, region)
    loc = pick_x0_indices(
        p_hat_c, target=target, topk=topk,
        min_p=target_min_p, candidate_idx=candidate_idx
    )
    if len(loc) == 0:
        raise ValueError(f"cluster {cluster_id} {region} 沒有可用中心點。")

    global_ids = [int(idx_c[pos]) for pos in loc]
    x_center = X_all_df.iloc[global_ids].to_numpy().mean(axis=0)
    model = behavior_models[int(cluster_id)]

    select_count = np.zeros(len(feature_names), dtype=np.int32)
    beta_list = {i: [] for i in range(len(feature_names))}
    run_tables = []

    for b in range(LOCAL_B):
        run_seed = int(base_seed + 1_000_000 * b)
        df_rounds, final_cols = run_Nround_fullp_screening(
            x0=x_center,
            blackbox_model=model,
            feature_names=feature_names,
            k=LOCAL_K,
            delta=LOCAL_DELTA,
            scaler=None,
            n_block=LOCAL_N_BLOCK,
            base_seed=run_seed + 1000 * int(cluster_id),
            thresholds=LOCAL_THRESHOLDS,
        )
        final_cols = np.asarray(final_cols, dtype=int)
        if len(df_rounds):
            df_rounds = df_rounds.copy()
            df_rounds["cluster"] = int(cluster_id)
            df_rounds["region"] = region
            df_rounds["x0_indices"] = [global_ids] * len(df_rounds)
            df_rounds["hatp_center"] = float(p_hat_c[loc].mean())
            df_rounds["p_final_center"] = float(model.predict_proba(x_center.reshape(1, -1))[:, 1][0])
            df_rounds["run_id"] = b
            run_tables.append(df_rounds)

        if final_cols.size == 0:
            continue
        select_count[final_cols] += 1
        if len(df_rounds):
            last_round = int(df_rounds["round_id"].max())
            df_last = df_rounds[df_rounds["round_id"] == last_round]
            for _, row in df_last[df_last["feature_idx"].isin(final_cols)][["feature_idx", "coef"]].iterrows():
                beta_list[int(row["feature_idx"])].append(float(row["coef"]))

    summary_rows = []
    for j in np.where(select_count > 0)[0]:
        betas = np.asarray(beta_list[int(j)], dtype=float)
        if betas.size == 0:
            continue
        se = float(betas.std(ddof=1) / np.sqrt(betas.size)) if betas.size >= 2 else np.nan
        summary_rows.append({
            "cluster": int(cluster_id),
            "region": region,
            "feature": feature_names[int(j)],
            "feature_idx": int(j),
            "select_count": int(select_count[int(j)]),
            "select_rate": float(select_count[int(j)] / LOCAL_B),
            "beta_n": int(betas.size),
            "beta_mean": float(betas.mean()),
            "beta_se": se,
            "beta_mean_minus_se": float(betas.mean() - se) if np.isfinite(se) else np.nan,
            "beta_mean_plus_se": float(betas.mean() + se) if np.isfinite(se) else np.nan,
        })

    df_summary = pd.DataFrame(summary_rows)
    if len(df_summary):
        df_summary = df_summary.sort_values(
            ["cluster", "select_count", "beta_mean"],
            ascending=[True, False, False],
        ).reset_index(drop=True)
    df_all_runs = pd.concat(run_tables, ignore_index=True) if len(run_tables) else pd.DataFrame()

    final_cols = (
        df_summary.query("select_rate == 1.0")["feature_idx"].to_numpy(dtype=int)
        if len(df_summary) else np.array([], dtype=int)
    )
    if final_cols.size == 0 and len(df_summary):
        final_cols = df_summary.sort_values(
            ["select_count", "beta_mean"], ascending=[False, False]
        ).head(10)["feature_idx"].to_numpy(dtype=int)
        print(f"Warning: cluster {cluster_id} {region} no select_rate==1.0 features; using top selected features.")
    if final_cols.size == 0:
        raise ValueError(f"cluster {cluster_id} {region} 沒有 final features。")

    info = local_surrogate_fixed_features_bigN(
        x0=x_center,
        blackbox_model=model,
        block_cols=final_cols,
        delta=LOCAL_DELTA,
        k=LOCAL_K,
        scaler=None,
        n_total=LOCAL_N_TOTAL,
        rng_seed=base_seed,
        center_first=True,
    )

    rows = [{
        "feature": "(intercept)",
        "feature_idx": -1,
        "coef": float(info["intercept"]),
        "se": np.nan,
        "t": np.nan,
        "p": float(info["intercept_p"]),
        "r2": float(info["r2"]),
    }]
    for t, col in enumerate(final_cols):
        rows.append({
            "feature": feature_names[int(col)],
            "feature_idx": int(col),
            "coef": float(info["coef_block"][t]),
            "se": float(info["se_block"][t]),
            "t": float(info["t_block"][t]),
            "p": float(info["p_block"][t]),
            "r2": float(info["r2"]),
        })
    coef_df = pd.DataFrame(rows).sort_values(["p"], ascending=True).reset_index(drop=True)

    center_original_indices = list(X_all_df.iloc[global_ids].index)
    center_rows = df_behavior_proba[
        (df_behavior_proba["cluster"] == int(cluster_id)) &
        (df_behavior_proba["original_index"].isin(center_original_indices))
    ]
    candidates = get_center_candidates(cluster_id, region)
    r2 = float(info.get("r2", np.nan))
    n_local = int(info.get("n", LOCAL_N_TOTAL))
    q = int(info.get("q", len(final_cols)))
    adj_r2 = 1.0 - (1.0 - r2) * (n_local - 1) / (n_local - q - 1) if n_local > q + 1 else np.nan

    local_model_summary_rows.append({
        "cluster": int(cluster_id),
        "region": region,
        "n_center_candidates": int(len(candidates)),
        "center_index": _display_value(center_original_indices),
        "center_true_y": _display_value(center_rows.sort_values("original_index")["true_y"].tolist()),
        "center_hatp": float(center_rows["hatp"].mean()) if len(center_rows) else np.nan,
        "n_local_samples": n_local,
        "local_positive_count": int(candidates["true_y"].sum()) if len(candidates) else 0,
        "local_positive_rate": float(candidates["true_y"].mean()) if len(candidates) else np.nan,
        "r2": r2,
        "adj_r2": float(adj_r2) if np.isfinite(adj_r2) else np.nan,
        "mae": float(info.get("mae", np.nan)) if np.isfinite(info.get("mae", np.nan)) else np.nan,
        "rmse": float(info.get("rmse", np.nan)) if np.isfinite(info.get("rmse", np.nan)) else np.nan,
        "n_features_used": int(len(final_cols)),
    })

    coef_part = coef_df[~coef_df["feature"].astype(str).str.lower().eq("(intercept)")].copy()
    coef_part["cluster"] = int(cluster_id)
    coef_part["region"] = region
    coef_part["center_index"] = _display_value(center_original_indices)
    coef_part["pvalue"] = coef_part["p"]
    coef_part["abs_tvalue"] = coef_part["t"].abs()
    coef_part["statistic"] = coef_part["t"]
    coef_part = coef_part.sort_values("abs_tvalue", ascending=False).reset_index(drop=True)
    coef_part["rank"] = np.arange(1, len(coef_part) + 1)
    coef_part = coef_part[[
        "cluster", "region", "center_index", "feature", "coef", "pvalue",
        "abs_tvalue", "statistic", "rank"
    ]]

    key = (int(cluster_id), region)
    local_model_coef_tables[key] = coef_part
    local_model_coef_rows.extend(coef_part.to_dict("records"))
    local_screening_tables[key] = {
        "df_all_runs": df_all_runs,
        "df_summary": df_summary,
        "coef_df": coef_df,
    }
    return coef_df


local_model_configs = [
    {"cluster": 0, "region": "high_risk", "topk": 3, "target": 0.95, "target_min_p": 0.95, "coef_file": "p=1_c0_final_B=50_topk=1_k=2_delta=01_n=100w.csv"},
    {"cluster": 0, "region": "decision_boundary", "topk": 10, "target": final_thresholds[0], "target_min_p": None, "coef_file": "p=02_c0_final_B=50_topk=10_k=2_delta=01_n=100w.csv"},
    {"cluster": 1, "region": "high_risk", "topk": 3, "target": 0.95, "target_min_p": 0.95, "coef_file": "p=06_c1_final_B=50_topk=2_k=2_delta=01_n=100w.csv"},
    {"cluster": 1, "region": "decision_boundary", "topk": 1, "target": final_thresholds[1], "target_min_p": None, "coef_file": "p=02_c1_final_B=50_topk=4_k=2_delta=01_n=100w.csv"},
]


for cfg in local_model_configs:
    coef_df = run_screening_for_local_model(
        cluster_id=cfg["cluster"],
        region=cfg["region"],
        topk=cfg["topk"],
        target=cfg["target"],
        target_min_p=cfg["target_min_p"],
        base_seed=RANDOM_STATE,
    )
    coef_df.to_csv(cfg["coef_file"], sep="\t", index=False)

local_model_summary_df = pd.DataFrame(local_model_summary_rows)
local_model_coef_df = pd.DataFrame(local_model_coef_rows)
local_model_summary_df.to_csv("local_model_summary_df.csv", index=False, encoding="utf-8-sig")
local_model_coef_df.to_csv("local_model_coef_df.csv", index=False, encoding="utf-8-sig")
display(local_model_summary_df)


,cluster,region,n_center_candidates,center_index,center_true_y,center_hatp,n_local_samples,local_positive_count,local_positive_rate,r2,adj_r2,mae,rmse,n_features_used
0,0,high_risk,3,"[289, 210, 392]","[1, 1, 1]",0.957196,2000000,3,1.0,0.612590,0.612574,NaN,NaN,84
1,0,decision_boundary,22,"[109, 176, 20, 134, 248, 198, 140, 105, 284, 184]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.132526,2000000,0,0.0,0.652156,0.652143,NaN,NaN,79
2,1,high_risk,3,"[1024, 1287, 1326]","[1, 1, 1]",0.959827,2000000,3,1.0,0.602489,0.602469,NaN,NaN,102
3,1,decision_boundary,1,1147,1,0.360926,2000000,1,1.0,0.641608,0.641584,NaN,NaN,135


# 14. Local model 結果整理


In [ ]:
display(local_model_summary_df)
display(local_model_coef_df.head(20))
print(local_model_coef_df.groupby(["cluster", "region"]).size())


# 15. 圖表輸出


In [35]:
def hist_count(
    data_list,
    label_list,
    bins=100,
    xlabel="預測機率",
    ylabel="數量",
    title=None,
    figsize=(8, 6),
    threshold_lines=None,
    save_path=None,
):
    plt.figure(figsize=figsize)
    for data, label in zip(data_list, label_list):
        p = pd.Series(data).dropna().values
        if len(p) == 0:
            continue
        plt.hist(p, bins=bins, alpha=0.6, label=label, edgecolor="black")
    if threshold_lines is not None:
        for threshold, label in threshold_lines:
            plt.axvline(threshold, color="red", linestyle="--", linewidth=1.8, label=label)
    plt.xticks(np.arange(0, 1.01, 0.2))
    plt.xlabel(xlabel, fontsize=14)
    plt.ylabel(ylabel, fontsize=14)
    if title is not None:
        plt.title(title)
    plt.legend()
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

df_final_proba = df_behavior_proba.rename(columns={"hatp": "final_proba"}).copy()

df_c0 = df_final_proba[df_final_proba["cluster"] == 0]
hist_count(
    data_list=[
        df_c0.loc[df_c0["true_y"] == 0, "final_proba"],
        df_c0.loc[df_c0["true_y"] == 1, "final_proba"],
    ],
    label_list=["群 0 真實良品", "群 0 真實不良品"],
    bins=100,
    xlabel="預測機率",
    ylabel="數量",
    threshold_lines=[(final_thresholds[0], "threshold = 0.15")],
    save_path="img/final_outputc0.png",
)

df_c1 = df_final_proba[df_final_proba["cluster"] == 1]
hist_count(
    data_list=[
        df_c1.loc[df_c1["true_y"] == 0, "final_proba"],
        df_c1.loc[df_c1["true_y"] == 1, "final_proba"],
    ],
    label_list=["群 1 真實良品", "群 1 真實不良品"],
    bins=80,
    xlabel="預測機率",
    ylabel="數量",
    threshold_lines=[(final_thresholds[1], "threshold = 0.37")],
    save_path="img/final_outputc1.png",
)


In [43]:
df_local_final = pd.read_csv(
    r"C:\Users\No\Documents\論文\notebooks\c1_high_risk.csv",
    sep="\t"
)

feature_col = "feature"   # 特徵名稱欄
coef_col = "coef"         # 係數欄
t_col = "t"               # t 統計量欄

# 轉成數值，避免空白或字串造成排序錯誤
df_local_final[coef_col] = pd.to_numeric(df_local_final[coef_col], errors="coerce")
df_local_final[t_col] = pd.to_numeric(df_local_final[t_col], errors="coerce")

# intercept
df_intercept = df_local_final[
    df_local_final[feature_col].astype(str).str.lower().eq("(intercept)")
]

# 非 intercept，依 |t| 由大到小取前 10 個特徵
df_feature_top10 = (
    df_local_final[
        ~df_local_final[feature_col].astype(str).str.lower().eq("(intercept)")
    ]
    .assign(abs_t=lambda d: d[t_col].abs())
    .sort_values("abs_t", ascending=False)
    .head(10)
)

# 合併：intercept + 10 個特徵，共 11 列
df_plot = pd.concat([df_intercept, df_feature_top10], axis=0).copy()
df_plot[feature_col] = df_plot[feature_col].replace("(intercept)", "intercept")

# y 軸由上而下：intercept, 統計量大到小的特徵
df_plot_for_bar = df_plot.iloc[::-1]

plt.figure(figsize=(8, 6))

# 正係數紅色，負係數藍色
colors = df_plot_for_bar[coef_col].apply(
    lambda x: "red" if x > 0 else "blue"
)

bars = plt.barh(
    df_plot_for_bar[feature_col],
    df_plot_for_bar[coef_col],
    color=colors
)

plt.axvline(0, linewidth=1, color="black")

# 文字位置：放在圖右側
x_text = df_plot_for_bar[coef_col].max() + 0.05

# 在每個長條右邊標係數值
for bar in bars:
    width = bar.get_width()
    y = bar.get_y() + bar.get_height() / 2

    text_color = "red" if width > 0 else "blue"

    plt.text(
        x_text,
        y,
        f"{width:.3f}",
        va="center",
        ha="left",
        fontsize=14,
        color=text_color
    )

plt.xlabel("係數", fontsize=14)
plt.ylabel("變數", fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.tight_layout()

output_path = r"C:\Users\No\Documents\論文\notebooks\c1_high_risk_coef.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

# 16. 最終檢查


In [37]:
required_objects = [
    "cluster_model_selection_df",
    "cluster_test_result_df",
    "overfit_check_df",
    "df_behavior_proba",
    "region_summary_df",
    "local_model_summary_df",
]
for obj_name in required_objects:
    if obj_name not in globals():
        raise NameError(f"Missing required dataframe: {obj_name}")

required_behavior_cols = [
    "original_index", "true_y", "hatp", "cluster", "selected_model", "selected_threshold",
    "high_risk_region", "decision_boundary_region",
]
missing_cols = [col for col in required_behavior_cols if col not in df_behavior_proba.columns]
if missing_cols:
    raise ValueError(f"df_behavior_proba missing columns: {missing_cols}")

if not np.isclose(final_thresholds[0], 0.15):
    raise ValueError("cluster 0 threshold is not 0.15")
if not np.isclose(final_thresholds[1], 0.37):
    raise ValueError("cluster 1 threshold is not 0.37")

if not (df_behavior_proba["high_risk_region"].to_numpy() == (df_behavior_proba["hatp"].to_numpy() >= 0.95)).all():
    raise ValueError("high_risk_region is inconsistent with hatp >= 0.95")

boundary_expected = (
    (df_behavior_proba["hatp"] >= df_behavior_proba["selected_threshold"] - 0.05) &
    (df_behavior_proba["hatp"] <= df_behavior_proba["selected_threshold"] + 0.05)
)
if not (df_behavior_proba["decision_boundary_region"].to_numpy() == boundary_expected.to_numpy()).all():
    raise ValueError("decision_boundary_region is inconsistent with selected threshold ? 0.05")

# Local surrogate uses behavior final model probability, hatp, not true_y.
assert "hatp" in df_behavior_proba.columns

notebook_source_text = Path("thesis.ipynb").read_text(encoding="utf-8") if Path("thesis.ipynb").exists() else ""
typo_checks = {
    "KNN" + "Imputater": "KNNImputer",
    "tra" + "ing": "training",
    "hat" + "_p": "hatp",
    "hatp" + "_final": "hatp",
}
for wrong, correct in typo_checks.items():
    if wrong in notebook_source_text:
        raise ValueError(f"Possible typo found: {wrong}; use {correct} or add a clear note.")

legacy_output_name = "out" + "puc0"
legacy_output_name_1 = "out" + "puc1"
if legacy_output_name in notebook_source_text or legacy_output_name_1 in notebook_source_text:
    print("Warning: old outpuc0/outpuc1 filenames still appear. Check thesis.tex references before renaming files.")

whole_plot_name = "final_" + "outputall.png"
if whole_plot_name in notebook_source_text:
    print("Warning: " + whole_plot_name + " still appears; current requirement keeps only cluster-specific final probability plots.")

print("完整流程執行完成")
print("模型評估：training OOF 選 threshold，testing fixed threshold 評估")
print("局部刻劃：behavior final model 的 hatp 作為 local surrogate 目標")
print("高度不良品預測區：hatp >= 0.95")
print("決策邊界區：selected threshold ± 0.05")


完整流程執行完成
模型評估：training OOF 選 threshold，testing fixed threshold 評估
局部刻劃：behavior final model 的 hatp 作為 local surrogate 目標
高度不良品預測區：hatp >= 0.95
決策邊界區：selected threshold ± 0.05
